In [ ]:
class Error_visualize:
    


    def __init__(self,df,Table_ALL,number):

        self.df = df
        self.Table_ALL = Table_ALL
        self.number = number


    def find_crossings(self,arr):
        a = arr[1:1000]
        arr_ex = np.append(a, 0)
        b  = arr * arr_ex
        cross_indices = np.where(b < 0)[0]
        value = 10/999 * cross_indices
        return cross_indices, value



    def cross_indices(self,l,DF_list): # number row index i = 0,1,2,....
    
        row_data = [item for item in l if item.get('row') == self.number]

        DF_0 = row_data[0]['DF_0']
        DF_1 = row_data[1]['DF_1']
        DF_2 = row_data[2]['DF_2']
        
        Diff_01 = DF_0 - DF_1
        Diff_12 = DF_1 - DF_2
        Diff_02 = DF_0 - DF_2

        c01,v01 = self.find_crossings(Diff_01)
        c12,v02 = self.find_crossings(Diff_12)
        c02,v03 = self.find_crossings(Diff_02)

        result = []
        result.append({'pair':'0-1','cross_indices':c01,'values':v01})
        result.append({'pair':'1-2','cross_indices':c12,'values':v02})
        result.append({'pair':'0-2','cross_indices':c02,'values':v03})
        

        # plot: 

        DF_diff1 = DF_list[self.number*3]['difference']
        DF_diff2 = DF_list[self.number*3 +1]['difference']
        DF_diff3 = DF_list[self.number*3 +2]['difference']
        
        fig, ax = plt.subplots(2,1,figsize=(20,10))

        sns.histplot(DF_diff1,color='blue',label='DF_0',kde=True,ax=ax[0])
        sns.histplot(DF_diff2,color='orange',label='DF_1',kde=True,ax=ax[0])
        sns.histplot(DF_diff3,color='green',label='DF_2',kde=True,ax=ax[0])

        # # 0-1の交点を描画
        # if len(v01) > 0:
        #     for x_val in v01:
        #         plt.axvline(x=x_val, color='red', linestyle='--', label='Cross 0-1 (x={:.3f})'.format(x_val))
                
        # # 1-2の交点を描画
        # if len(v02) > 0:
        #     for x_val in v02:
        #         # 凡例が多重表示されるのを防ぐため、2つ目以降の線のlabelをNoneにする
        #         plt.axvline(x=x_val, color='purple', linestyle='--', label=None) 
        #         # 最初の線にのみラベルを付ける
        #         if v02[0] == x_val:
        #             plt.axvline(x=x_val, color='purple', linestyle='--', label=f'Cross 1-2 (x={x_val:.3f})')


        # # 0-2の交点を描画 (別の色で)
        # if len(v03) > 0:
        #     for x_val in v03:
        #         plt.axvline(x=x_val, color='brown', linestyle='--', label=None) 
        #         if v03[0] == x_val:
        #             plt.axvline(x=x_val, color='brown', linestyle='--', label=f'Cross 0-2 (x={x_val:.3f})')
        
        ax[0].set_xlim(0,10)
        # ax[0].set_ylim(0,100)
        ax[0].legend()
        ax[0].set_title(f'Row {self.number} Density Plot with Cross Points')
    


    
        combined_data = pd.DataFrame({
                        'Correct Predictions': DF_diff1,
                        f'Errors with Label': DF_diff2,
                        'Errors with other Labels': DF_diff3
                    })
        x_range = np.linspace(0,10,1000)
        ax[1].plot(x_range, DF_0, color='blue', label='DF_0 KDE', linewidth=2)
        ax[1].plot(x_range, DF_1, color='orange', label='DF_1 KDE', linewidth=2)
        ax[1].plot(x_range, DF_2, color='green', label='DF_2 KDE', linewidth=2)
        
        # 0-1の交点を描画
        if len(v01) > 0:
            for x_val in v01:
                ax[1].axvline(x=x_val, color='red', linestyle='--', label='Cross 0-1 (x={:.3f})'.format(x_val))
                
        # 1-2の交点を描画
        if len(v02) > 0:
            for x_val in v02:
                # 凡例が多重表示されるのを防ぐため、2つ目以降の線のlabelをNoneにする
                ax[1].axvline(x=x_val, color='purple', linestyle='--', label=None) 
                # 最初の線にのみラベルを付ける
                if v02[0] == x_val:
                    ax[1].axvline(x=x_val, color='purple', linestyle='--', label=f'Cross 1-2 (x={x_val:.3f})')


        # 0-2の交点を描画 (別の色で)
        if len(v03) > 0:
            for x_val in v03:
                ax[1].axvline(x=x_val, color='brown', linestyle='--', label=None) 
                if v03[0] == x_val:
                    ax[1].axvline(x=x_val, color='brown', linestyle='--', label=f'Cross 0-2 (x={x_val:.3f})')

        
        plt.legend()
        ax[1].set_title(f'Row {self.number} KDE Plot with Cross Points')
        plt.tight_layout()
        plt.show()





        return result 

    
    def crosspoint(self,cross_indices_list):


        indice_list = []
        for i in range(len(cross_indices_list)):
            for j in range(len(cross_indices_list[i]['values'])):
                indice_list.append(cross_indices_list[i]['values'][j])
        upper = max(indice_list)
        lower = min(indice_list)

        return lower,upper 

    def plot_error_visualize(self,lower,upper,DF_list):

        DF_diff1 = DF_list[self.number*3]
        DF_diff2 = DF_list[self.number*3 +1]
        DF_diff3 = DF_list[self.number*3 +2]
        
        DF_ALL = pd.concat([DF_diff2,DF_diff3],ignore_index=True)

        row = self.Table_ALL.iloc[self.number]
        pred = row['pred']
        label = row['label']

        self.df['error_combi'] = (self.df['pred'] == pred) & (self.df['label'] == label)&(self.df['error']==True)

        fig,ax = plt.subplots(3,1,figsize=(10,10))
        gridtable = pd.pivot_table(index = 'y',columns='x',values='error_combi',data=self.df)
        ax[0].imshow(gridtable, cmap='Reds')
        gridtable = pd.pivot_table(index = 'y',columns='x',values='label',data=self.df)
        ax[0].imshow(gridtable, cmap='viridis', alpha=0.3)

        self.df['error_combi'] = (self.df['pred'] == pred)&(self.df['error']==True)
        gridtable_2 = pd.pivot_table(index = 'y',columns='x',values='error_combi',data=self.df)

        ax[1].imshow(gridtable_2, cmap='Reds')
        ax[1].imshow(gridtable,cmap='viridis', alpha=0.3)
        ax[1].set_title('All Errors for Pred: {}'.format(pred))
        ax[0].set_title(f'Error Visualization for Row {self.number} (Pred: {pred}, Label: {label})\nError Range: {lower:.3f} to {upper:.3f}')

        # Error by Border 

        DF_ALL['error_zone']=DF_ALL['difference'].apply(lambda x: 0 if x< lower else (1 if x<=upper else 2))
        grid_table_3 = pd.pivot_table(index = 'y',columns='x',values='error_zone',data=DF_ALL)
        im = ax[2].imshow(grid_table_3, cmap='viridis', vmin=0, vmax=2)


        cmap = get_cmap('viridis')
        # normalizeしていないので、0, 0.5, 1.0 の位置の色を取るイメージです
        # または単純に viridis(0.0), viridis(0.5), viridis(1.0) を指定します
        colors = [cmap(0.0), cmap(0.5), cmap(1.0)] 
        labels = [f'Zone 0 (< {lower:.3f})', f'Zone 1 ({lower:.3f}~{upper:.3f})', f'Zone 2 (> {upper:.3f})']

        patches = [mpatches.Patch(color=colors[i], label=labels[i]) for i in range(3)]


        ax[2].legend(handles=patches, loc='upper right', bbox_to_anchor=(1.5, 1))

        plt.show()

In [ ]:
analyzer = Error_visualize(df,TABLE_ALL,0)
cross_indices_list = analyzer.cross_indices(l,DF_list)
lower,upper = analyzer.crosspoint(cross_indices_list)
analyzer.plot_error_visualize(lower,upper,DF_list)